# Privacy, Governance & Compliance — NovaCred Credit Applications

## Purpose
This notebook examines the NovaCred dataset through a data governance and regulatory compliance lens. It identifies all PII fields, demonstrates pseudonymization, maps findings to GDPR provisions, references the EU AI Act, and proposes concrete governance controls.

## Dataset
- **Source:** `../data/cleaned_credit_applications.csv`
- **Context:** Credit application records from NovaCred fintech platform

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re

df = pd.read_csv('../data/cleaned_credit_applications.csv')

print(f'Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns')
print(f'\nAll columns:')
for col in df.columns:
    print(f'  - {col}')

---

## 2. PII Field Identification

Personally Identifiable Information (PII) refers to any data that can be used to identify a specific individual, either directly or in combination with other fields.

We classify PII into three categories:
- **Direct identifiers**: uniquely identify a person on their own (e.g. name, SSN)
- **Quasi-identifiers**: identify a person when combined with other fields (e.g. zip code + DOB)
- **Sensitive behavioral data**: reveals patterns that can infer protected characteristics

In [ ]:
pii_fields = {
    'applicant_info.full_name':     ('Direct identifier',         'High'),
    'applicant_info.email':         ('Direct identifier',         'High'),
    'applicant_info.ssn':           ('Sensitive identifier',      'Critical'),
    'applicant_info.ip_address':    ('Quasi-identifier',          'Medium'),
    'applicant_info.date_of_birth': ('Quasi-identifier',          'Medium'),
    'applicant_info.zip_code':      ('Quasi-identifier',          'Low'),
    'spending_behavior':            ('Sensitive behavioral data', 'Medium-High'),
}

print('PII Field Identification')
print('=' * 80)
print(f'{"Field":<40} {"PII Type":<30} {"Risk":<12} {"Present in Dataset"}')
print('-' * 80)

for field, (pii_type, risk) in pii_fields.items():
    present = field in df.columns
    non_null = df[field].notna().sum() if present else 'N/A'
    status = f'{non_null} records' if present else 'NOT FOUND'
    print(f'{field:<40} {pii_type:<30} {risk:<12} {status}')

print(f'\nTotal PII fields identified: {len(pii_fields)}')

### Note on `spending_behavior`

`spending_behavior` is flagged as **sensitive behavioral data**. While not a direct identifier, it reveals patterns about a person's financial habits and lifestyle, which can be used to infer protected characteristics (e.g. religion, health status). Under GDPR, behavioral data that carries this inferential risk should be treated with the same caution as traditional PII.

---

## 3. Pseudonymization — SHA-256 Hash of SSN

**Pseudonymization** replaces identifying data with a substitute so the person cannot be directly identified, while still allowing records to be linked if the original value is known.

It is **not the same as anonymization**: SHA-256 is a one-way function, but if someone has the original SSN they can hash it and match it to the dataset. The link is not permanently destroyed.

GDPR **Art. 4(5)** explicitly recognizes pseudonymization as a valid risk-reduction measure.

In [ ]:
# Show original SSN values (sample)
print('BEFORE pseudonymization — sample SSN values:')
print(df['applicant_info.ssn'].head(5).to_string())

# Apply SHA-256 hashing
df['ssn_hashed'] = df['applicant_info.ssn'].apply(
    lambda x: hashlib.sha256(str(x).encode()).hexdigest() if pd.notna(x) else np.nan
)

print('\nAFTER pseudonymization — SHA-256 hashed values:')
print(df['ssn_hashed'].head(5).to_string())

hashed_count = df['ssn_hashed'].notna().sum()
print(f'\nTotal records pseudonymized: {hashed_count}')
print(f'Hash length (chars): {len(df["ssn_hashed"].dropna().iloc[0])}')
print('\nNote: original SSN column should be dropped before sharing or storing the dataset.')

In [ ]:
# Drop original SSN column
df_pseudonymized = df.drop(columns=['applicant_info.ssn'])

print(f'Original column count:  {df.shape[1]}')
print(f'After dropping SSN:     {df_pseudonymized.shape[1]}')
print('applicant_info.ssn removed: True')

---

## 4. GDPR Mapping

The table below maps our findings to the relevant GDPR provisions.

| GDPR Article | Requirement | Finding in NovaCred Dataset |
|---|---|---|
| **Art. 6 — Lawful Basis** | Processing must have a legal basis | Credit scoring qualifies under contractual necessity or legitimate interest |
| **Art. 5 — Minimisation + Storage Limitation** | Only collect necessary data; don't keep longer than needed | `ip_address`, `zip_code`, `spending_behavior` — necessity should be justified; no retention policy is currently defined |
| **Art. 17 — Right to Erasure** | Users can request deletion of their data | No erasure mechanism is demonstrated in the current data pipeline |
| **Art. 22 — Automated Decision-Making** | Right not to be subject to purely automated decisions with significant effects | Credit scoring is a textbook Art. 22 case — no consent tracking field exists in the dataset |
| **Art. 4(5) — Pseudonymization** | Recognized as a valid risk-reduction measure | Demonstrated via SHA-256 hash of SSN (see Section 3) |

### Art. 22 — Standalone Note

**Art. 22** is important enough to address on its own. It directly restricts automated decision-making that produces significant legal effects — which credit scoring clearly does. NovaCred must either obtain explicit consent, rely on contractual necessity, or implement suitable safeguards. This is not just a control to implement; it is a legal gate that must be cleared before the system can operate at all.

**Compliance gap identified:** the current dataset contains no field tracking whether applicant consent was collected for automated decision-making. This must be addressed before deployment.

---

## 5. EU AI Act — High-Risk Classification

Credit scoring is listed under **Annex III, point 5(b)** of the EU AI Act as a high-risk AI application. This triggers several obligations:

- Transparency and explainability of decisions
- Human oversight mechanisms
- Technical documentation and conformity assessment
- Monitoring for discriminatory outcomes

> **Link to bias analysis:** The fairness metrics computed in `02-bias-analysis.ipynb` (Disparate Impact ratio, Demographic Parity difference) are directly relevant here. Under the EU AI Act, high-risk systems must demonstrate they do not produce discriminatory outcomes across protected groups (gender, age). Those findings should be reviewed as part of any conformity assessment — the formal audit a company must pass before deploying a high-risk AI system.

---

## 6. Proposed Governance Controls

### Audit Trail
Every model decision should be automatically logged with a timestamp, the input features used, and the output (approved/rejected). This creates a traceable history that allows NovaCred to explain any decision if a customer challenges it or a regulator requests an audit.

### Human Oversight
Borderline cases — such as applicants with a credit score right at the approval threshold — should be flagged for manual review instead of receiving a fully automated decision. This is required by both the EU AI Act for high-risk systems and GDPR Art. 22, which protects individuals from purely algorithmic decisions that significantly affect them.

### Consent
Before processing data through an automated system, applicants must be clearly informed that an algorithm will evaluate their application and give explicit consent. Under GDPR Art. 22, automated decision-making with significant effects (like a loan rejection) is only lawful under specific conditions, including the data subject's consent. **Importantly, the current dataset contains no field tracking whether consent was collected — this is a compliance gap that must be addressed before deployment.**

### Retention Policy
NovaCred must define a maximum retention period (e.g. 5 years), after which all PII fields must be deleted or fully anonymized. This is required by GDPR Art. 5's storage limitation principle, which prohibits keeping personal data longer than necessary for its original purpose.

---

## 7. Summary

| Area | Status | Gap Identified |
|---|---|---|
| PII Identification | Done | `spending_behavior` requires justification |
| Pseudonymization | Done | Original SSN must be dropped before sharing |
| GDPR Compliance | Partial | No consent tracking, no erasure mechanism, no retention policy |
| EU AI Act | High-risk system | Conformity assessment not yet performed |
| Governance Controls | Proposed | Audit trail, human oversight, consent, retention — all pending implementation |